# Gravitational Lens ML Round Template

Change `ROUND_NAME` and `ROUND_SCRIPT`, attach the matching Kaggle Dataset, then run top to bottom.

In [ ]:
import os, subprocess, textwrap
ROUND_NAME = "phase4_v0_1"
ROUND_SCRIPT = "scripts/phase4_v0_1_round.py"
REPO_URL = "https://github.com/dasbaq/GV.git"
REPO_DIR = "/kaggle/working/repo"
print(subprocess.check_output(["nvidia-smi"], text=True))
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd /kaggle/working/repo/Gravitational_Lens_MultiMode

In [ ]:
from pathlib import Path
h5s = sorted(Path('/kaggle/input').rglob('*.h5'))
pkls = sorted(Path('/kaggle/input').rglob('*.pkl'))
print('H5 files:')
for p in h5s: print(' ', p)
print('PKL files:')
for p in pkls: print(' ', p)
train = next(p for p in h5s if p.name == f'{ROUND_NAME}.h5')
unfiltered = next((p for p in h5s if p.name == f'{ROUND_NAME}_eval_unfiltered.h5'), None)
scaler = next((p for p in pkls if p.name == f'target_scaler_{ROUND_NAME}.pkl'), None)
equivalence = next((p for p in sorted(Path('/kaggle/input').rglob('*equivalence.json')) if ROUND_NAME in p.name), None)
os.environ['LENS_DATA_PATH'] = str(train)
if unfiltered: os.environ['LENS_DATA_PATH_UNFILTERED'] = str(unfiltered)
if scaler: os.environ['LENS_SCALER_PATH'] = str(scaler)
os.environ['LENS_WORK_ROOT'] = '/kaggle/working'
print('LENS_DATA_PATH=', os.environ['LENS_DATA_PATH'])
print('LENS_DATA_PATH_UNFILTERED=', os.environ.get('LENS_DATA_PATH_UNFILTERED'))
print('LENS_SCALER_PATH=', os.environ.get('LENS_SCALER_PATH'))
print('equivalence=', equivalence)

In [ ]:
# Short CUDA sanity run. Acceptance/leak triggers are gated off below 10 epochs.
eq_arg = f"--equivalence-from {equivalence}" if equivalence else ""
!python {ROUND_SCRIPT} --phase train {eq_arg} --device cuda --workers 4 --epochs 2 --bootstrap-n 0

In [ ]:
# Full run.
!python {ROUND_SCRIPT} --phase train {eq_arg} --device cuda --workers 4 --epochs 50 --bootstrap-n 1000

In [ ]:
import json
for p in sorted(Path('/kaggle/working/logs').glob('*.json')):
    print('\n==', p.name, '==')
    data = json.loads(p.read_text())
    if 'stage_b_acceptance_report' in data:
        print(json.dumps(data['stage_b_acceptance_report'], indent=2)[:4000])
    elif 'best' in data:
        print(json.dumps(data.get('best', {}), indent=2)[:4000])
    else:
        print(json.dumps(data, indent=2)[:2000])